# GNU Radio IQ Processing Pipeline

A complete SDR workflow: capture IQ data, balance I/Q components, compute Welch PSD, remove DC spike, and plot results.

### General Pseudocode

```
1. IMPORT LIBRARIES (numpy, scipy.signal, matplotlib)
2. iq = GENERATE SYNTHETIC IQ (sample_rate=2.4e6, duration=0.01, signal_type="sine")
3. iq_balanced = BALANCE IQ COMPONENTS (iq)
4. f, psd = APPLY WELCH METHOD (iq_balanced, fs=2.4e6, nperseg=1024)
5. iq_cleaned = REMOVE DC SPIKE (iq_balanced)
6. f_clean, psd_clean = APPLY WELCH METHOD (iq_cleaned, fs=2.4e6, nperseg=1024)
7. PLOT results (f, psd, f_clean, psd_clean)
```

### Pipeline Flow

```
Synthetic IQ -> Balance IQ -> Welch PSD -> DC Spike Remove -> Welch PSD -> Plot
```

---
## Section 1: Setup & Requirements

Install dependencies from `reference-nb-requirements.txt` and import core libraries. All subsequent sections depend on these imports.

```
FUNCTION InstallRequirements(requirements_path)
    DECLARE subprocess AS MODULE
    <- subprocess.run(["pip", "install", "-r", requirements_path])
    RETURN None
END FUNCTION

FUNCTION ImportLibraries()
    DECLARE np AS MODULE
    DECLARE signal AS MODULE
    DECLARE plt AS MODULE
    <- IMPORT numpy AS np
    <- IMPORT scipy.signal
    <- IMPORT matplotlib.pyplot AS plt
    RETURN (np, signal, plt)
END FUNCTION
```

In [ ]:
import subprocess
import sys


def install_requirements(requirements_path: str) -> None:
    """Install Python packages from a requirements file.

    :param requirements_path: Path to the requirements.txt file.
    :type requirements_path: str
    :return: None
    """
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", requirements_path],
        check=True,
        capture_output=True,
    )


install_requirements("reference-nb-requirements.txt")

In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt


print(f"numpy:      {np.__version__}")
print(f"scipy:      {signal.__name__}")
print(f"matplotlib: {plt.matplotlib.__version__}")

---
## Section 2: Generate Synthetic IQ Data

Generate a synthetic IQ signal for testing the pipeline. The `sine` type produces a clean 1 kHz tone; `noise` produces random complex Gaussian samples.

```
FUNCTION GenerateSyntheticIQ(sample_rate, duration, signal_type)
    DECLARE t AS ARRAY of FLOAT
    DECLARE iq AS ARRAY of COMPLEX64
    t <- LINSPACE(0, duration, sample_rate * duration)
    IF signal_type = "sine" THEN
        iq <- EXP(1j * 2 * PI * 1000 * t)
    ELSE IF signal_type = "noise" THEN
        iq <- RANDN(len(t)) + 1j * RANDN(len(t))
    END IF
    RETURN iq.astype(COMPLEX64)
END FUNCTION
```

In [ ]:
def generate_synthetic_iq(
    sample_rate: float = 2.4e6,
    duration: float = 0.01,
    signal_type: str = "sine",
) -> np.ndarray:
    """Generate synthetic IQ data for pipeline testing.

    :param sample_rate: Sampling rate in Hz. Default 2.4e6.
    :type sample_rate: float
    :param duration: Signal duration in seconds. Default 0.01.
    :type duration: float
    :param signal_type: One of "sine" or "noise". Default "sine".
    :type signal_type: str
    :return: Complex IQ array of shape (L,) where L = sample_rate * duration.
    :rtype: np.ndarray
    """
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

    if signal_type == "sine":
        iq = np.exp(1j * 2 * np.pi * 1000 * t)
    elif signal_type == "noise":
        iq = np.random.randn(len(t)) + 1j * np.random.randn(len(t))
    else:
        raise ValueError(f"Unknown signal_type: {signal_type}")

    return iq.astype(np.complex64)


SAMPLE_RATE = 2.4e6
DURATION = 0.01

iq_raw = generate_synthetic_iq(SAMPLE_RATE, DURATION, "sine")
print(f"IQ shape: {iq_raw.shape}, dtype: {iq_raw.dtype}")
print(f"Mean power: {np.mean(np.abs(iq_raw)**2):.4f}")

---
## Section 3: Balance I/Q Components

Remove DC offset from I and Q components separately. This ensures the signal is centered around zero in the complex plane, which is required for accurate spectral analysis.

```
FUNCTION BalanceIQ(iq)
    DECLARE I AS ARRAY of FLOAT
    DECLARE Q AS ARRAY of FLOAT
    DECLARE I_balanced AS ARRAY of FLOAT
    DECLARE Q_balanced AS ARRAY of FLOAT
    I <- REAL(iq)
    Q <- IMAG(iq)
    I_balanced <- I - MEAN(I)
    Q_balanced <- Q - MEAN(Q)
    RETURN I_balanced + 1j * Q_balanced
END FUNCTION
```

In [ ]:
def balance_iq(iq: np.ndarray) -> np.ndarray:
    """Remove DC offset from I and Q components.

    Subtracts the mean from each component independently,
    centering the signal around the origin.

    :param iq: Complex IQ signal.
    :type iq: np.ndarray
    :return: DC-balanced complex IQ signal.
    :rtype: np.ndarray
    """
    i_comp = iq.real - np.mean(iq.real)
    q_comp = iq.imag - np.mean(iq.imag)
    return (i_comp + 1j * q_comp).astype(np.complex64)


iq_balanced = balance_iq(iq_raw)
print(f"Before balance - I mean: {np.mean(iq_raw.real):.6f}, Q mean: {np.mean(iq_raw.imag):.6f}")
print(f"After balance  - I mean: {np.mean(iq_balanced.real):.6f}, Q mean: {np.mean(iq_balanced.imag):.6f}")

---
## Section 4: Welch PSD Estimation

Compute Power Spectral Density using Welch's method. This averages overlapping windowed segments to reduce variance in the spectral estimate.

```
FUNCTION ApplyWelchMethod(iq, fs, nperseg, noverlap)
    DECLARE f AS ARRAY of FLOAT
    DECLARE psd AS ARRAY of FLOAT
    (f, psd) <- scipy.signal.welch(iq, fs=fs, nperseg=nperseg, noverlap=noverlap)
    RETURN (f, psd)
END FUNCTION
```

In [ ]:
def apply_welch_method(
    iq: np.ndarray,
    fs: float = 2.4e6,
    nperseg: int = 1024,
    noverlap: int = 512,
) -> tuple:
    """Compute PSD using Welch's method.

    :param iq: Complex IQ signal.
    :type iq: np.ndarray
    :param fs: Sampling frequency in Hz. Default 2.4e6.
    :type fs: float
    :param nperseg: Length of each segment. Default 1024.
    :type nperseg: int
    :param noverlap: Number of overlapping points. Default 512.
    :type noverlap: int
    :return: Tuple of (frequency array, PSD array).
    :rtype: tuple[np.ndarray, np.ndarray]
    """
    f, psd = signal.welch(iq, fs=fs, nperseg=nperseg, noverlap=noverlap)
    return f, psd


NPERSEG = 1024
NOVERLAP = 512

f_raw, psd_raw = apply_welch_method(iq_balanced, SAMPLE_RATE, NPERSEG, NOVERLAP)
print(f"Frequency bins: {f_raw.shape}, PSD bins: {psd_raw.shape}")
print(f"Peak frequency: {f_raw[np.argmax(psd_raw)]:.0f} Hz")

---
## Section 5: DC Spike Removal

Remove DC spike using median subtraction. Median is robust to outliers and strong tones, making it more effective than mean subtraction for DC spike removal in the presence of interfering signals.

```
FUNCTION RemoveDCSpike(iq)
    DECLARE i_comp AS ARRAY of FLOAT
    DECLARE q_comp AS ARRAY of FLOAT
    DECLARE i_clean AS ARRAY of FLOAT
    DECLARE q_clean AS ARRAY of FLOAT
    i_comp <- REAL(iq)
    q_comp <- IMAG(iq)
    i_clean <- i_comp - MEDIAN(i_comp)
    q_clean <- q_comp - MEDIAN(q_comp)
    RETURN i_clean + 1j * q_clean
END FUNCTION
```

In [ ]:
def remove_dc_spike(iq: np.ndarray) -> np.ndarray:
    """Remove DC spike using median subtraction.

    More robust than mean subtraction when strong tones
    are present in the signal.

    :param iq: Complex IQ signal.
    :type iq: np.ndarray
    :return: DC-spike-free complex IQ signal.
    :rtype: np.ndarray
    """
    i_clean = iq.real - np.median(iq.real)
    q_clean = iq.imag - np.median(iq.imag)
    return (i_clean + 1j * q_clean).astype(np.complex64)


iq_cleaned = remove_dc_spike(iq_balanced)
print(f"Before DC removal - DC component: {np.abs(np.mean(iq_balanced)):.6f}")
print(f"After DC removal  - DC component: {np.abs(np.mean(iq_cleaned)):.6f}")

In [ ]:
f_clean, psd_clean = apply_welch_method(iq_cleaned, SAMPLE_RATE, NPERSEG, NOVERLAP)
print(f"Cleaned PSD peak frequency: {f_clean[np.argmax(psd_clean)]:.0f} Hz")

---
## Section 6: Plot Results

Compare Welch PSD before and after DC spike removal. The left panel shows the raw spectrum with DC spike; the right panel shows the cleaned spectrum.

```
FUNCTION PlotResults(f, psd, f_clean, psd_clean)
    DECLARE fig AS FIGURE
    DECLARE ax1, ax2 AS AXIS
    (fig, (ax1, ax2)) <- SUBPLOTS(1, 2)
    ax1.PLOT(f, psd)
    ax1.SET_TITLE("Before DC Removal")
    ax2.PLOT(f_clean, psd_clean)
    ax2.SET_TITLE("After DC Removal")
    SAVEFIG("output/welch_comparison.png")
    RETURN fig
END FUNCTION
```

In [ ]:
def plot_results(
    f: np.ndarray,
    psd: np.ndarray,
    f_clean: np.ndarray,
    psd_clean: np.ndarray,
) -> plt.Figure:
    """Plot Welch PSD before and after DC spike removal.

    :param f: Frequency array (before cleanup).
    :type f: np.ndarray
    :param psd: PSD array (before cleanup).
    :type psd: np.ndarray
    :param f_clean: Frequency array (after cleanup).
    :type f_clean: np.ndarray
    :param psd_clean: PSD array (after cleanup).
    :type psd_clean: np.ndarray
    :return: Matplotlib Figure object.
    :rtype: plt.Figure
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(f / 1e3, 10 * np.log10(psd))
    ax1.set_title("Before DC Removal")
    ax1.set_xlabel("Frequency (kHz)")
    ax1.set_ylabel("PSD (dB/Hz)")
    ax1.grid(True)

    ax2.plot(f_clean / 1e3, 10 * np.log10(psd_clean))
    ax2.set_title("After DC Removal")
    ax2.set_xlabel("Frequency (kHz)")
    ax2.set_ylabel("PSD (dB/Hz)")
    ax2.grid(True)

    fig.suptitle("Welch PSD Comparison", fontsize=14)
    fig.tight_layout()
    fig.savefig("output/welch_comparison.png", dpi=150)

    return fig


_ = plot_results(f_raw, psd_raw, f_clean, psd_clean)